In [46]:
# import packages 
import pandas as pd 
import numpy as np 
import mlflow
from mlflow.tracking import MlflowClient    
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials


import matplotlib.pyplot as plt 
from sklearn.tree import plot_tree

from scipy import stats
import statsmodels.api as sm

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve, auc


## What you will find in this notebook: 

- ML Models to train this data
    2. Decision Tree 
    3. Random Forest 
    4. AdaBoost 
    
- An intuitive and mathematical explanation of each of these models. 

In [18]:
# read csv file 

heart_df = pd.read_csv("../data/cardio_train.csv", delimiter =";")

heart_df

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,19240,2,168,76.0,120,80,1,1,1,0,1,0
69996,99995,22601,1,158,126.0,140,90,2,2,0,0,1,1
69997,99996,19066,2,183,105.0,180,90,3,1,0,1,0,1
69998,99998,22431,1,163,72.0,135,80,1,2,0,0,0,1


In [19]:
heart_df.dtypes

id               int64
age              int64
gender           int64
height           int64
weight         float64
ap_hi            int64
ap_lo            int64
cholesterol      int64
gluc             int64
smoke            int64
alco             int64
active           int64
cardio           int64
dtype: object

In [20]:
categorical_col = ['cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']


heart_df[categorical_col] = heart_df[categorical_col].astype('category')
heart_df.dtypes

id                int64
age               int64
gender            int64
height            int64
weight          float64
ap_hi             int64
ap_lo             int64
cholesterol    category
gluc           category
smoke          category
alco           category
active         category
cardio         category
dtype: object

#### There is no missing data, refer to analysis.ipynb

## Fix Data 

In [21]:
average_days_per_year = 365.25

heart_df['age'] = (heart_df['age']/ average_days_per_year).round().astype(int)

heart_df

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,50,2,168,62.0,110,80,1,1,0,0,1,0
1,1,55,1,156,85.0,140,90,3,1,0,0,1,1
2,2,52,1,165,64.0,130,70,3,1,0,0,0,1
3,3,48,2,169,82.0,150,100,1,1,0,0,1,1
4,4,48,1,156,56.0,100,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,53,2,168,76.0,120,80,1,1,1,0,1,0
69996,99995,62,1,158,126.0,140,90,2,2,0,0,1,1
69997,99996,52,2,183,105.0,180,90,3,1,0,1,0,1
69998,99998,61,1,163,72.0,135,80,1,2,0,0,0,1


In [22]:
heart_df["bmi"] = (heart_df["weight"]/((heart_df["height"]/100) **2)).round(2)

heart_df.pop('id')
heart_df.pop('height')
heart_df.pop('weight')

heart_df

,age,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,bmi
0,50,2,110,80,1,1,0,0,1,0,21.97
1,55,1,140,90,3,1,0,0,1,1,34.93
2,52,1,130,70,3,1,0,0,0,1,23.51
3,48,2,150,100,1,1,0,0,1,1,28.71
4,48,1,100,60,1,1,0,0,0,0,23.01
...,...,...,...,...,...,...,...,...,...,...,...
69995,53,2,120,80,1,1,1,0,1,0,26.93
69996,62,1,140,90,2,2,0,0,1,1,50.47
69997,52,2,180,90,3,1,0,1,0,1,31.35
69998,61,1,135,80,1,2,0,0,0,1,27.10


In [23]:
heart_df= heart_df[["age",
                    "gender",
                    "bmi",
                    "ap_hi",
                    "ap_lo",
                    "cholesterol",
                    "gluc",
                    "smoke",
                    "alco",
                    "active",
                    "cardio"]]
heart_df

,age,gender,bmi,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,50,2,21.97,110,80,1,1,0,0,1,0
1,55,1,34.93,140,90,3,1,0,0,1,1
2,52,1,23.51,130,70,3,1,0,0,0,1
3,48,2,28.71,150,100,1,1,0,0,1,1
4,48,1,23.01,100,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
69995,53,2,26.93,120,80,1,1,1,0,1,0
69996,62,1,50.47,140,90,2,2,0,0,1,1
69997,52,2,31.35,180,90,3,1,0,1,0,1
69998,61,1,27.10,135,80,1,2,0,0,0,1


## Set MLFlow 

In [24]:
mlflow.__version__

'2.15.1'

In [25]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('cardio_disease')

<Experiment: artifact_location='/Users/bera/Desktop/MSDS/603/mlops/experiments/cardio_disease/mlruns/1', creation_time=1742880844134, experiment_id='1', last_update_time=1742880844134, lifecycle_stage='active', name='cardio_disease', tags={}>

## Setup the model 

In [26]:
y = heart_df["cardio"]
X = heart_df.drop(columns=["cardio"])


In [32]:
def train_val_test(X, y):
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, train_size=0.7, random_state=12, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.15, random_state=12, stratify=y_train_full
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = train_val_test(X, y)


In [33]:
# Check how stratified the data is. 
print(f"Y Train Data:\n{pd.Series(y_train).value_counts()/y_train.shape}")
print(f"\nY Test Data:\n{pd.Series(y_test).value_counts()/y_test.shape}")


Y Train Data:
cardio
0    0.500312
1    0.499688
Name: count, dtype: float64

Y Test Data:
cardio
0    0.500286
1    0.499714
Name: count, dtype: float64


## Define the models 

In [34]:
param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 5, None]
}

param_grid_rf = {    
    'n_estimators': [50, 100],
    'max_depth': [2, 5, None],
    'bootstrap': [True, False]    
}

param_grid_gb = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [2, 3, 5]
}

models_to_train = [
    ("DecisionTree", DecisionTreeClassifier(random_state=42), param_grid_dt),
    ("RandomForest", RandomForestClassifier(random_state=42), param_grid_rf),
    ("GradientBoost", GradientBoostingClassifier(random_state=42), param_grid_gb)
]

In [37]:

train_df = pd.DataFrame(np.column_stack((X_train, y_train)), 
                        columns=[f"feature_{i}" for i in range(X_train.shape[1])] + ["target"])
val_df = pd.DataFrame(np.column_stack((X_val, y_val)), 
                      columns=[f"feature_{i}" for i in range(X_val.shape[1])] + ["target"])
test_df = pd.DataFrame(np.column_stack((X_test, y_test)), 
                       columns=[f"feature_{i}" for i in range(X_test.shape[1])] + ["target"])

train_csv_path = "../datasets_artifacts/train_cardio.csv"
val_csv_path   = "../datasets_artifacts/val_cardio.csv"
test_csv_path  = "../datasets_artifacts/test_cardio.csv"

train_df.to_csv(train_csv_path, index=False)
val_df.to_csv(val_csv_path, index=False)
test_df.to_csv(test_csv_path, index=False)


In [39]:
best_runs = []

for model_name, estimator, param_grid in models_to_train:
    with mlflow.start_run(run_name=model_name) as run:
        grid_search = GridSearchCV(
            estimator, param_grid,
            scoring='accuracy',
            cv=3,
            n_jobs=-1,
            verbose=1
        )
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        y_val_pred = best_model.predict(X_val)
        val_accuracy = accuracy_score(y_val, y_val_pred)

        mlflow.log_params(grid_search.best_params_)
        mlflow.log_metric("val_accuracy", val_accuracy)

        mlflow.sklearn.log_model(best_model, artifact_path=f"{model_name}_model")
    
        best_runs.append((model_name, run.info.run_id, val_accuracy, best_model))




Fitting 3 folds for each of 8 candidates, totalling 24 fits


2025/03/24 22:52:07 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


Fitting 3 folds for each of 12 candidates, totalling 36 fits


2025/03/24 22:52:17 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


Fitting 3 folds for each of 18 candidates, totalling 54 fits


2025/03/24 22:52:33 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


## Choose Best three models 

In [42]:
best_runs

[('DecisionTree',
  '528771ddc80c49158a4d1497fe0d1e21',
  0.7360544217687075,
  DecisionTreeClassifier(max_depth=5, random_state=42)),
 ('RandomForest',
  '33a0213a85424a2db6ef8f5de3602ce2',
  0.7280272108843537,
  RandomForestClassifier(max_depth=5, random_state=42)),
 ('GradientBoost',
  '8a05e63af4e1425e97947aca3b5a2720',
  0.7389115646258504,
  GradientBoostingClassifier(max_depth=5, n_estimators=50, random_state=42))]

## Final Best Model

In [44]:
best_runs_sorted = sorted(best_runs, key=lambda x: x[2], reverse=True)
best_model_name, best_run_id, best_val_acc, best_estimator_obj = best_runs_sorted[0]

print("Final Best Model")
print("Model Name:", best_model_name)
print("Validation Accuracy:", best_val_acc)
print("Run ID:", best_run_id)


Final Best Model
Model Name: GradientBoost
Validation Accuracy: 0.7389115646258504
Run ID: 8a05e63af4e1425e97947aca3b5a2720


## Evaluate Model 


In [45]:
final_y_test_pred = best_estimator_obj.predict(X_test)
final_test_accuracy = accuracy_score(y_test, final_y_test_pred)

print(f"Final Test Accuracy ({best_model_name}): {final_test_accuracy:.4f}")

Final Test Accuracy (GradientBoost): 0.7329


In [48]:
client = MlflowClient()
client.set_tag(run_id=best_run_id, key="final_model", value="true")
client.log_metric(run_id=best_run_id, key="test_accuracy", value=final_test_accuracy)


In [50]:
model_name_in_registry = "my_best_classification_model"

model_uri = f"runs:/{best_run_id}/{best_model_name}_model"
model_details = mlflow.register_model(
    model_uri=model_uri,
    name=model_name_in_registry
)


Successfully registered model 'my_best_classification_model'.
Created version '1' of model 'my_best_classification_model'.


In [52]:
model_version = model_details.version
client.transition_model_version_stage(
    name=model_name_in_registry,
    version=model_version,
    stage="Staging", 
    archive_existing_versions=False
)

print(f"Registered model '{model_name_in_registry}', version: {model_version}, stage: 'Staging'")
print("Final Test Accuracy:", final_test_accuracy)

Registered model 'my_best_classification_model', version: 1, stage: 'Staging'
Final Test Accuracy: 0.732904761904762


/var/folders/50/7ys27s856cj0_7svprhcb3580000gp/T/ipykernel_5450/3291941084.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
